# Exercise 3.2 
This exercise show how to create a very simple Bayesian Network using pgmpy and asking a simple query using variable elimination

First we need to import the modules that we will use and then to create a new Graph (G).

In [3]:
#This code is modified from the examples at pgmpy.org Copyright (c) 2013-2021 pgmpy
from pgmpy.models import BayesianNetwork
from pgmpy.factors.discrete.CPD import TabularCPD
G = BayesianNetwork()

Once we have a graph we can add some nodes a,b and c

In [4]:
G.add_node(node='a')
G.add_node(node='b')
G.add_node(node='c')

We can then add the edges directed from a to b and a to c

In [5]:
G.add_edge(u='a',v='b')
G.add_edge(u='a',v='c')

Next we need a CPT at each node
For this we are going to use TabularCPD. This reques the following arguments:
- the name of the variable whose CPT is being defined.
- no. of possible states of the variable
- the probability values for the CPD table. 
- list of variables in evidences(if any) w.r.t. which CPD is defined.
- no. of states of the evidence variables (if any)

Below we define CPTs for P(a) (root node, therefore no evidence variables), P(b|a) (evidence is variable a) and P(c|a) (evidence is variable a)
Each of our variables just has 2 states (0 or 1).


In [6]:
cpd_a = TabularCPD('a', 2, [[0.2], [0.8]])
cpd_b = TabularCPD('b', 2, [[0.3, 0.7], [0.7, 0.3]],
                   evidence=['a'],
                   evidence_card=[2])
cpd_c = TabularCPD('c', 2, [[0.1, 0.9], [0.9, 0.1]],
                   evidence=['a'],
                   evidence_card=[2])
G.add_cpds(cpd_a, cpd_b, cpd_c)

It is worth checking the model to ensure it has been defined correctly.
You can also view the nodes, edges and independencies if you require.

In [7]:
# Checking if the cpds are valid for the model
G.check_model()

True

In [8]:
# Viewing nodes of the model
G.nodes()

NodeView(('a', 'b', 'c'))

In [9]:
G.edges()

OutEdgeView([('a', 'b'), ('a', 'c')])

In [10]:
#Checking independcies of a node
G.local_independencies('b')

(b ⟂ c | a)

In [11]:
#Listing all Independencies
G.get_independencies()

(c ⟂ b | a)
(b ⟂ c | a)

Once you are satisfied that the model is correct we can perform our queries on the model.
In this example we are asking "what is the probability P(c|a=1)?".
Try out some other queries on the network. Such as P(b|a=0) and P(a|b=1,c=1)

In [15]:
# Infering the posterior probability
from pgmpy.inference import VariableElimination

infer = VariableElimination(G)
posterior_p = infer.query(['c'], evidence={'a': 1})
print(posterior_p)
posterior_p = infer.query(['b'], evidence={'a' : 0})
print(posterior_p)
posterior_p = infer.query(['a'], evidence={'b' : 1, 'c' : 1})
print(posterior_p)

+------+----------+
| c    |   phi(c) |
+======+==========+
| c(0) |   0.9000 |
+------+----------+
| c(1) |   0.1000 |
+------+----------+
+------+----------+
| b    |   phi(b) |
+======+==========+
| b(0) |   0.3000 |
+------+----------+
| b(1) |   0.7000 |
+------+----------+
+------+----------+
| a    |   phi(a) |
+======+==========+
| a(0) |   0.8400 |
+------+----------+
| a(1) |   0.1600 |
+------+----------+
